In [1]:
import xarray as xr
import numpy as np
import pandas as pd

from snobedo.snotel import SnotelLocations
from snobedo.snotel import SnotelLocations, CsvParser

from snobedo.lib.dask_utils import start_cluster, client_ip_and_port

from pathlib import Path, PurePath

import matplotlib.pyplot as plt

import os
import glob

In [2]:
client = start_cluster(4, 24)
client_ip_and_port(client)

2025-06-26 13:25:19,122 - tornado.application - ERROR - Exception in callback functools.partial(<bound method IOLoop._discard_future_result of <tornado.platform.asyncio.AsyncIOMainLoop object at 0x7ff159de2690>>, <Task finished name='Task-16' coro=<SpecCluster._correct_state_internal() done, defined at /uufs/chpc.utah.edu/common/home/u1037042/software/pkg/miniconda3/envs/snow_viz/lib/python3.12/site-packages/distributed/deploy/spec.py:346> exception=RuntimeError('Command exited with non-zero exit code.\nExit code: 1\nCommand:\nsbatch /scratch/local/u1037042/4935410/tmphwvosaz_.sh\nstdout:\n\nstderr:\nsbatch: error: Batch job submission failed: Invalid qos specification\n\n')>)
Traceback (most recent call last):
  File "/uufs/chpc.utah.edu/common/home/u1037042/software/pkg/miniconda3/envs/snow_viz/lib/python3.12/site-packages/tornado/ioloop.py", line 750, in _run_callback
    ret = callback()
          ^^^^^^^^^^
  File "/uufs/chpc.utah.edu/common/home/u1037042/software/pkg/miniconda3/e

10.242.76.198:8787


In [3]:
SHARED_STORE = PurePath('/uufs/chpc.utah.edu/common/home/skiles-group1')
DATA_DIR = SHARED_STORE.joinpath('jmeyer')
SNOTEL_DIR = DATA_DIR.joinpath( 'Snotel')
from pathlib import Path, PurePath
snotel_sites = SnotelLocations()
snotel_sites.load_from_json(SNOTEL_DIR / 'site-locations/snotel_sites.json')
snotel_sites.Irwin

SnotelSite(name='Irwin', lon=[317131], lat=[4306375])

In [4]:
from dask.distributed import Client

# Optional: start Dask client to monitor computation
client = Client()

# file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit/toposplit_*.nc'
# file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_2022/toposplit_*.nc'
file_pattern = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/hrrr_toposplit_2023/toposplit_*.nc'

# file_pattern = f"{SHARED_STORE_RUN}/erw_spires_dsw3_dlwrf/{water_year}/erw_50m/run*/net_solar.nc"

HRRR_solar = xr.open_mfdataset(
    file_pattern,
    combine='by_coords',
    parallel=True, chunks={'time': 24}, # 'y' :10, 'x': 10},
)

/uufs/chpc.utah.edu/common/home/u1037042/software/pkg/miniconda3/envs/snow_viz/lib/python3.12/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 41995 instead
  warnings.warn(


In [5]:
# SOS
sos_path = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/station-data/sos-isfs/hourlyAvg_radiation_uw.nc'
# sos_single = '/uufs/chpc.utah.edu/common/home/u1037042/olson/snow-data/station-data/sos-isfs/sos_isfs_qc_geo_tiltcor_5min_v20241227/isfs_sos_qc_geo_tiltcor_5min_v2_20230212.nc'
ds = xr.open_dataset(sos_path)

In [6]:
sos_coords = {"lat": ds.latitude_uw.isel(time=0).values.item(), "lon": ds.longitude_uw.isel(time=0).values.item()}

print(sos_coords)

{'lat': 38.941712, 'lon': -106.97328799999998}


In [ ]:
#SOS coords
{'lat': 38.941712, 'lon': 38.941712}

In [7]:
import geopandas as gpd
from shapely.geometry import Point

def convert_to_utm(coords):
    """
    Convert coordinates from WGS84 (lat/lon) to UTM Zone 13N (EPSG:26913)
    
    :param coords: Dictionary with 'lat' and 'lon' keys
    :return: Dictionary with 'utm_x' and 'utm_y' keys for UTM coordinates
    """
    # Create a GeoDataFrame with the coordinates
    gdf = gpd.GeoDataFrame(
        {'geometry': [Point(coords['lon'], coords['lat'])]}, 
        crs="EPSG:4326"  # WGS84 (Lat/Lon)
    )

    # Convert to UTM Zone 13N
    gdf_utm = gdf.to_crs(epsg=26913)

    # Extract the UTM X and Y coordinates
    utm_x, utm_y = gdf_utm.geometry.x[0], gdf_utm.geometry.y[0]
    
    # Return the result as a dictionary
    return {'utm_x': utm_x, 'utm_y': utm_y}

# Define m1_coords and s3_coords (example data)
m1_coords = {"lat": 38.95615768432617, "lon": -106.98785400390625}
s3_coords = {"lat": 39.750715, "lon": -104.998649}
sos_coords = {"lat": 38.941712, "lon": -106.97328799999998}

# Convert both coordinates
m1_utm = convert_to_utm(m1_coords)
s3_utm = convert_to_utm(s3_coords)
sos_utm = convert_to_utm(sos_coords)

# Combine the results into a final object
final_utm_coords = {
    'm1_coords_utm': m1_utm, #ARM QCRAD also at M1 site! (CHECK)
    's3_coords_utm': s3_utm,
    'sos_coords_utm': sos_utm
}

# Print the result
print(final_utm_coords)


{'m1_coords_utm': {'utm_x': np.float64(327754.55108678306), 'utm_y': np.float64(4313790.299013529)}, 's3_coords_utm': {'utm_x': np.float64(500115.76987025986), 'utm_y': np.float64(4400089.488481599)}, 'sos_coords_utm': {'utm_x': np.float64(328982.05598244135), 'utm_y': np.float64(4312159.612842733)}}


In [8]:
# Subset once using .sel with method='nearest'
irwin_data = HRRR_solar.sel(x=snotel_sites.Irwin.lon, y=snotel_sites.Irwin.lat, method='nearest')

# Drop unnecessary dimensions
irwin_data = irwin_data.squeeze(['x', 'y'])

# Now compute all at once
irwin_data = irwin_data[['ghi', 'dsw1', 'dsw3', 'dsw3h', 'k']].compute()

# Access variables
irwin_ghi = irwin_data['ghi']
irwin_dsw1 = irwin_data['dsw1']
irwin_dsw30 = irwin_data['dsw3']   # Assuming `dsw3` is the same as `dsw30`
irwin_dsw3 = irwin_data['dsw3h']
irwin_k = irwin_data['k']

In [9]:
# Define helper function to extract and compute data at one site
def extract_site_data(site_key):
    x = final_utm_coords[site_key]['utm_x']
    y = final_utm_coords[site_key]['utm_y']
    
    site_ds = HRRR_solar.sel(x=x, y=y, method='nearest')[['ghi', 'dsw1', 'dsw3', 'dsw3h', 'k']]
    return site_ds.squeeze().compute()

# Extract for M1
m1_data = extract_site_data('m1_coords_utm')
m1_ghi = m1_data['ghi']
m1_dsw1 = m1_data['dsw1']
m1_dsw30 = m1_data['dsw3']
m1_dsw3 = m1_data['dsw3h']
m1_k = m1_data['k']

# Extract for S3
s3_data = extract_site_data('s3_coords_utm')
s3_ghi = s3_data['ghi']
s3_dsw1 = s3_data['dsw1']
s3_dsw30 = s3_data['dsw3']
s3_dsw3 = s3_data['dsw3h']
s3_k = s3_data['k']

# Extract for SOS
sos_data = extract_site_data('sos_coords_utm')
sos_ghi = sos_data['ghi']
sos_dsw1 = sos_data['dsw1']
sos_dsw30 = sos_data['dsw3']
sos_dsw3 = sos_data['dsw3h']
sos_k = sos_data['k']


In [10]:
import os

# Create folder if it doesn't already exist
save_dir = 'modeled-station-2023'
os.makedirs(save_dir, exist_ok=True)

# S3 datasets
# s3_obs.to_netcdf(os.path.join(save_dir, 's3_obs.nc'))
# s3_sail.to_netcdf(os.path.join(save_dir, 's3_sail.nc'))
s3_dsw3.to_netcdf(os.path.join(save_dir, 's3_dsw3.nc'))
s3_k.to_netcdf(os.path.join(save_dir, 's3_k.nc'))
s3_dsw1.to_netcdf(os.path.join(save_dir, 's3_dsw1.nc'))
s3_ghi.to_netcdf(os.path.join(save_dir, 's3_ghi.nc'))
s3_dsw30.to_netcdf(os.path.join(save_dir, 's3_dsw30.nc'))

# M1 datasets
# m1_obs.to_netcdf(os.path.join(save_dir, 'm1_obs.nc'))
# m1_sail.to_netcdf(os.path.join(save_dir, 'm1_sail.nc'))
m1_dsw3.to_netcdf(os.path.join(save_dir, 'm1_dsw3.nc'))
m1_k.to_netcdf(os.path.join(save_dir, 'm1_k.nc'))
m1_dsw1.to_netcdf(os.path.join(save_dir, 'm1_dsw1.nc'))
m1_ghi.to_netcdf(os.path.join(save_dir, 'm1_ghi.nc'))
m1_dsw30.to_netcdf(os.path.join(save_dir, 'm1_dsw30.nc'))

# Irwin datasets
irwin_dsw3.to_netcdf(os.path.join(save_dir, 'irwin_dsw3.nc'))
irwin_k.to_netcdf(os.path.join(save_dir, 'irwin_k.nc'))
irwin_dsw1.to_netcdf(os.path.join(save_dir, 'irwin_dsw1.nc'))
irwin_ghi.to_netcdf(os.path.join(save_dir, 'irwin_ghi.nc'))
irwin_dsw30.to_netcdf(os.path.join(save_dir, 'irwin_dsw30.nc'))

# SOS
sos_dsw3.to_netcdf(os.path.join(save_dir, 'sos_dsw3.nc'))
sos_k.to_netcdf(os.path.join(save_dir, 'sos_k.nc'))
sos_dsw1.to_netcdf(os.path.join(save_dir, 'sos_dsw1.nc'))
sos_ghi.to_netcdf(os.path.join(save_dir, 'sos_ghi.nc'))
sos_dsw30.to_netcdf(os.path.join(save_dir, 'sos_dsw30.nc'))